In [1]:
%matplotlib inline

import os
import sys
import math
import itertools

from sys import platform

sys.path.append('../../')

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken

import torch
import torch.nn as nn


%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel
from llm_from_scratch.CH2.text_data_set import create_dataloader_v1
from llm_from_scratch.CH5.loss import calc_loss_batch
from llm_from_scratch.CH5.utils import find_highest_gradient
from llm_from_scratch.CH5.optim import evaluate_model, generate_and_print_sample, train_model
print(f"torch version: {version('torch')}")

torch version: 2.12.0


In [2]:
# define a grid of hyperparameters to search over
HPARAM_GRID={
    "warmup_iters":[10,20,30],
    "drop_rate":[0., 0.1, 0.2],
    "min_lr":[0.00005, 0.00001, 0.0001]
}

# generate hyper-parameter combinations
hyperparameter_combinations=list(itertools.product(*HPARAM_GRID.values()))
total_combinations=len(hyperparameter_combinations)
print(f"Total combinations {total_combinations}: {hyperparameter_combinations}")

# place holder for the best loss and best hyperparameters
best_val_loss=float('inf')
best_hparams={}

filepath='../the-verdict.txt'
with open(filepath, 'r', encoding='utf-8') as f: text_data=f.read()
tokenizer=tiktoken.get_encoding('gpt2')
device=torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.mps.is_available() else torch.device('cpu')

train_ratio=0.95
split_idx=int(train_ratio*len(text_data))

torch.manual_seed(123)
interrupted=False
current_config=0
for combination in hyperparameter_combinations:

    try:
        current_config+=1
        print(f"Evaluating configuration {current_config} of {total_combinations}")

        # unpack the current combination of hyperparameters
        HPARAM_CONFIG=dict(zip(HPARAM_GRID.keys(), combination))

        GPT_CONFIG_124M={
            "vocab_size":50257,                    # vocabulary size
            "context_length":256,                  # context-length --shortened from original 1024 tokens
            "emb_dim":768,                         # embedding dimension
            "n_heads":12,                          # number of attention heads
            "n_layers":12,                         # number of layers
            "drop_rate":HPARAM_CONFIG['drop_rate'],
            "qkv_bias":False                       # query-key-value bias
        }
        train_loader=create_dataloader_v1(text_data[:split_idx], batch_size=2, max_length=GPT_CONFIG_124M['context_length'],
                                         stride=GPT_CONFIG_124M['context_length'], drop_last=True, shuffle=True, num_workers=0)
        val_loader=create_dataloader_v1(text_data[split_idx:], batch_size=2, max_length=GPT_CONFIG_124M['context_length'],
                                       stride=GPT_CONFIG_124M['context_length'], drop_last=False, shuffle=False, num_workers=0)

        model=GPTModel(GPT_CONFIG_124M)
        model.to(device)

        optimizer=torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)
        train_model(model, train_loader, val_loader, optimizer, device, n_epochs=10, eval_iter=1, 
                                         warmup_steps=HPARAM_CONFIG['warmup_iters'], initial_lr=0.0001, min_lr=HPARAM_CONFIG['min_lr'],
                                         eval_freq=5, start_context='Nevertheless', tokenizer=tokenizer)
        train_loss, val_loss=evaluate_model(model, train_loader, val_loader=val_loader, device=device)
        # log the best hyperparameters based on validation loss
        if val_loss<best_val_loss:
            best_val_loss=val_loss
            best_train_loss=train_loss
            best_hparams=HPARAM_CONFIG
        
    except KeyboardInterrupt:
        print("Hyperparameter search completed")
        print(f"Best hyperparameters: {best_hparams}")
        print(f'Best Val loss: {best_val_loss} | Traing loss {train_loss}')
        interrupted=True
        break

if not interrupted:
    print("Hyperparameter search completed")
    print(f"Best hyperparameters: {best_hparams}")
    print(f'Best Val loss: {best_val_loss} | Traing loss {train_loss}')

Total combinations 27: [(10, 0.0, 5e-05), (10, 0.0, 1e-05), (10, 0.0, 0.0001), (10, 0.1, 5e-05), (10, 0.1, 1e-05), (10, 0.1, 0.0001), (10, 0.2, 5e-05), (10, 0.2, 1e-05), (10, 0.2, 0.0001), (20, 0.0, 5e-05), (20, 0.0, 1e-05), (20, 0.0, 0.0001), (20, 0.1, 5e-05), (20, 0.1, 1e-05), (20, 0.1, 0.0001), (20, 0.2, 5e-05), (20, 0.2, 1e-05), (20, 0.2, 0.0001), (30, 0.0, 5e-05), (30, 0.0, 1e-05), (30, 0.0, 0.0001), (30, 0.1, 5e-05), (30, 0.1, 1e-05), (30, 0.1, 0.0001), (30, 0.2, 5e-05), (30, 0.2, 1e-05), (30, 0.2, 0.0001)]
Evaluating configuration 1 of 27
EP 1 (Iter 000000): Train loss 10.568 Val loss 10.690
EP 1 (Iter 000005): Train loss 8.315 Val loss 8.624
Nevertheless,,,, the,,,,,,,,,, the,,,,,, the,,,,,,,,, the,,,,,,,,,,,,,,,,,
EP 2 (Iter 000010): Train loss 6.125 Val loss 6.754
EP 2 (Iter 000015): Train loss 5.773 Val loss 6.764
Nevertheless the the the the the the the the the the. I-- the the the the. I. I-- the the the the the-- the the the the the the the the the the the the the the-- t